# Scalable indicator calculations with Dask

This tutorial demonstrates how to run indicator calculations on larger datasets using **Dask** for lazy evaluation and parallel computation.

> **Key Concept**: Before continuing with this hands-on tutorial, we strongly recommend reading the concept page on [Scalability and performance](../concepts/scalability_performance.rst), which explains Dask chunking principles, spatial vs. temporal chunking, task vs. peer-to-peer rechunking engines, and worker memory management.

Datasets loaded via **earthkit-data** (e.g. from GRIB or NetCDF files) can be passed directly to indicators or converted to lazy chunked arrays.


In [ ]:
import earthkit.data as ekd

import earthkit.climate as ekc

## 1. Loading a chunked lazy dataset

When working with large climate datasets (e.g. ERA5 or CMIP6), data arrays are loaded as **dask-backed DataArrays**.
Here we load a 3D spatial-temporal daily maximum temperature dataset (`tasmax`) backed by Dask arrays from `earthkit-climate-sample`.


In [ ]:
tasmax = ekd.from_source("earthkit-climate-sample", "synthetic-daily-dask-temperature").to_xarray()
tasmax

## 2. Computing indicators lazily

When passing a dask-backed DataArray to **earthkit-climate** indicator functions, the index calculation is evaluated **lazily**.
The function returns a new DataArray containing a Dask computational task graph without performing heavy numerical computation immediately.


In [ ]:
hot_days_lazy = ekc.indicators.tx_days_above(tasmax, thresh="300 K", freq="YS")
hot_days_lazy

## 3. Triggering computation and saving results

To execute the task graph across available CPU threads, call `.compute()` or write directly to a NetCDF/Zarr store using `to_netcdf` or `to_zarr`.


In [ ]:
hot_days_computed = hot_days_lazy.compute()
hot_days_computed

## 4. Groupby considerations for daily climatologies and percentiles

Operations involving daily climatologies or percentile thresholds require contiguous time series for each spatial point.

* **Best practice**: Ensure the time dimension is unchunked (or chunked in multi-year blocks) before computing daily percentiles or climatologies.
* **Rechunking**: If your disk data is time-sliced (e.g. 1 day per chunk), rechunk spatially: `tasmax.chunk({"time": -1, "lat": 5, "lon": 5})`.


In [ ]:
tasmax_rechunked = tasmax.chunk({"time": -1, "lat": 5, "lon": 5})
per90_lazy = ekc.utils.climatology.rolling_percentiles(tasmax_rechunked, p=90, window_width=5)
per90_lazy